# Cycle 1 — Preprocessing: `premier_league_matches.csv`

**Project:** Football Predictor — Match Outcome Prediction (Win / Draw / Loss)  
**Dataset:** `premier_league_matches.csv`  
**Depends on:** `cycle1_exploration_premier_league_matches.ipynb` (read that first)

---

## Purpose of this Notebook

This notebook takes the raw `premier_league_matches.csv` and prepares it for machine learning. Every issue identified during exploration is addressed here step by step. The output is a clean, model-ready dataset saved to `data/processed/`.

**Steps covered:**
1. Load the raw data
2. Drop useless columns
3. Reconstruct the correct 3-class target variable (H/D/A)
4. Drop goal columns (data leakage)
5. Encode form columns (HM1–HM5, AM1–AM5)
6. Encode team names
7. Extract season from date, then drop date
8. Convert matchweek to integer
9. Encode the target variable to numbers
10. Final check and save

---
## Key Findings from Exploration (Summary)

Before preprocessing, here is a reminder of every issue identified during exploration and what we are doing about it:

| Issue | Found In | Fix |
|---|---|---|
| `FTR` is binary (H/NH) — not 3-class | Exploration Cell 4 | Reconstruct from `FTHG`/`FTAG` |
| `FTHG`/`FTAG` leak the result | Exploration Cell 2 | Drop after reconstructing `FTR` |
| `Unnamed: 0` is a redundant index | Exploration Cell 1 | Drop immediately |
| `HTFormPtsStr`/`ATFormPtsStr` are string duplicates | Exploration Cell 2 | Drop — numeric versions already exist |
| `HM1–HM5`/`AM1–AM5` contain `M` (missing) | Exploration Cell 6 | Encode W=3, D=1, L=0, M=0 |
| `HomeTeam`/`AwayTeam` are strings | Exploration Cell 3 | Label encode to integers |
| `Date` is a string, not datetime | Exploration Cell 3 | Parse, extract Season, then drop |
| `MW` is float64, should be int | Exploration Cell 3 | Convert to int |
| Features have different scales | Exploration Cell 7 | Note for modelling — apply StandardScaler for LogReg |

---
## Step 1 — Load the Raw Data

**What it does:** Loads the raw dataset and confirms it matches what we explored.

**Why:** Always reload from raw in preprocessing — never depend on state left over from another notebook.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/premier_league_matches.csv')
print('Shape:', df.shape)
df.head()

### Output
- Shape: **(6840, 40)**

### Observations
- Confirmed — matches the exploration notebook
- All 40 columns present, ready for cleaning

---
## Step 2 — Drop Useless Columns

**What it does:** Removes columns that are either redundant, carry no predictive value, or are string representations of information already captured numerically.

**Why:** Keeping unnecessary columns adds noise and increases dimensionality without benefit.

**Columns dropped:**
- `Unnamed: 0` — a row index, not a feature
- `HTFormPtsStr` — string version of form (e.g. "WWDLL"), already captured numerically in `HTFormPts`
- `ATFormPtsStr` — same as above for the away team

In [ ]:
cols_to_drop = ['Unnamed: 0', 'HTFormPtsStr', 'ATFormPtsStr']
df = df.drop(columns=cols_to_drop)

print('Shape after dropping useless columns:', df.shape)
print('Remaining columns:', df.columns.tolist())

### Output
- Shape: **(6840, 37)** — dropped 3 columns
- Remaining columns no longer include `Unnamed: 0`, `HTFormPtsStr`, `ATFormPtsStr`

### Observations
- No rows were lost — only columns removed
- All remaining columns are either the target, identifiers, or actual features

---
## Step 3 — Reconstruct the Correct Target Variable (FTR)

**What it does:** Replaces the broken binary `FTR` column (H/NH) with a correct 3-class version (H/D/A) derived from the actual goals scored.

**Why:** This is the most critical fix in the entire preprocessing pipeline. The original `FTR` column only had two values — Home Win and Not Home — because the Kaggle author converted it to a binary problem. Our project requires 3 classes: Home Win, Draw, Away Win.

**Logic:**
- `FTHG > FTAG` → H (Home Win)
- `FTHG < FTAG` → A (Away Win)
- `FTHG == FTAG` → D (Draw)

In [ ]:
df['FTR'] = df.apply(
    lambda row: 'H' if row['FTHG'] > row['FTAG'] else ('A' if row['FTHG'] < row['FTAG'] else 'D'),
    axis=1
)

print('Reconstructed FTR distribution:')
print(df['FTR'].value_counts())

### Output
```
FTR
H    3176
A    1913
D    1751
```

### Observations
- We now have all 3 correct classes
- Home wins are the most common (3,176 — 46.4%) — confirms home advantage
- Away wins: 1,913 (28.0%)
- Draws: 1,751 (25.6%)
- Total: 3176 + 1913 + 1751 = 6840 ✓ — no rows lost

### Class Imbalance Note
Home wins are nearly twice as common as draws or away wins. This is a real pattern in football data, not an error. However, it means the model may be biased towards predicting Home Wins. This will be addressed in the modelling notebook using balanced class weights.

### Notes for Report
- The original dataset had a pre-existing flaw that required reconstruction of the target variable
- This is a good example of why raw data must always be inspected before use
- The reconstruction is reliable because `FTHG` and `FTAG` are objective facts (goals scored)

---
## Step 4 — Drop Goal Columns (Data Leakage)

**What it does:** Removes `FTHG` (Full Time Home Goals) and `FTAG` (Full Time Away Goals) from the dataset.

**Why:** These columns were only kept temporarily to reconstruct `FTR`. Now that the target is fixed, they must be dropped. Keeping them would be data leakage — the model would learn to predict the result directly from the scoreline, which is not available before the match.

In [ ]:
df = df.drop(columns=['FTHG', 'FTAG'])

print('Shape after dropping goal columns:', df.shape)

### Output
- Shape: **(6840, 35)** — dropped 2 more columns

### Observations
- `FTHG` and `FTAG` are gone — no leakage risk from these columns
- The only information remaining about the match outcome is in `FTR` (the target)

---
## Step 5 — Encode Form Columns (HM1–HM5, AM1–AM5)

**What it does:** Converts the last 5 match result columns from string values (W/D/L/M) to numbers.

**Why:** Machine learning models cannot work with string values — every feature must be numeric.

**Encoding used:**
- `W` = 3 (a win earns 3 points in football)
- `D` = 1 (a draw earns 1 point)
- `L` = 0 (a loss earns 0 points)
- `M` = 0 (no previous match — treated as neutral, same as a loss)

**Why this encoding?** Using actual point values (3/1/0) preserves the real-world meaning of the results — a win is worth 3 times more than a draw. This is more informative than arbitrary numbering like W=2, D=1, L=0.

In [ ]:
result_map = {'W': 3, 'D': 1, 'L': 0, 'M': 0}
form_cols = ['HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5']

for col in form_cols:
    df[col] = df[col].map(result_map)

print('Form columns after encoding:')
print(df[form_cols].head(10))
print()
print('Any NaN introduced?', df[form_cols].isnull().sum().sum())

### Output
```
Form columns after encoding:
   HM1  HM2  HM3  HM4  HM5  AM1  AM2  AM3  AM4  AM5
0    0    0    0    0    0    0    0    0    0    0
1    0    0    0    0    0    0    0    0    0    0
...

Any NaN introduced? 0
```

### Observations
- First rows show all 0s — expected, as these are early season matches (Matchweek 1) where no prior results exist
- No NaN values introduced — all W/D/L/M values were successfully mapped
- The encoding preserves the football point system (3/1/0) which is more meaningful than arbitrary labels

### Note
- `M` is encoded as 0 (same as a loss) — this is a simplification. An alternative would be to encode `M` as the average (1.33) but 0 is simpler and commonly used

---
## Step 6 — Encode Team Names

**What it does:** Converts `HomeTeam` and `AwayTeam` from team name strings to integer labels.

**Why:** ML models require numeric input. Team names must be converted to numbers. The same encoding is applied to both columns so that, for example, Arsenal = 0 in both `HomeTeam` and `AwayTeam`.

**Method:** Label encoding — each unique team name gets a unique integer. Teams are sorted alphabetically first so the mapping is consistent and reproducible.

In [ ]:
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()
team_map = {team: idx for idx, team in enumerate(sorted(all_teams))}

df['HomeTeam'] = df['HomeTeam'].map(team_map)
df['AwayTeam'] = df['AwayTeam'].map(team_map)

print('Number of unique teams:', len(team_map))
print()
print('Team encoding (first 10):')
print(dict(list(team_map.items())[:10]))
print()
print('HomeTeam and AwayTeam after encoding:')
print(df[['HomeTeam', 'AwayTeam']].head())

### Output
```
Number of unique teams: 44

Team encoding (first 10):
{'Arsenal': 0, 'Aston Villa': 1, 'Birmingham': 2, 'Blackburn': 3, 'Blackpool': 4,
 'Bolton': 5, 'Bournemouth': 6, 'Brighton': 7, 'Burnley': 8, 'Cardiff': 9}

HomeTeam and AwayTeam after encoding:
   HomeTeam  AwayTeam
0        11        24
1        12        41
2        13        27
...
```

### Observations
- 44 unique teams across 18 seasons — Premier League has relegation/promotion so teams change each year
- The same integer represents the same team whether they are home or away — consistent encoding
- No NaN values — all team names were successfully mapped

### Important Note for Modelling
Label encoding assigns arbitrary integers to teams (Arsenal=0, Aston Villa=1, etc.). This implies a false ordering — the model might interpret Arsenal (0) as "less than" Aston Villa (1), which has no real meaning. A better approach is one-hot encoding, but with 44 teams that creates 44 extra columns. For tree-based models (Random Forest, XGBoost), label encoding works fine. For Logistic Regression, one-hot encoding would be more appropriate.

### Notes for Report
- 44 teams across 18 seasons reflects the promotion/relegation system of the Premier League
- Team identity is an important feature — Manchester City in 2017 is very different from Hull City in 2017

---
## Step 7 — Extract Season from Date, Then Drop Date

**What it does:** Parses the `Date` column into a proper datetime, extracts the season year as a new feature, then drops the raw date.

**Why:** The raw date string is not useful as a feature. However, the season number is useful — it captures long-term trends (e.g. the game has evolved tactically over 18 years). A season starts in August, so any match before August is counted as the previous year's season.

**Season logic:** If month >= 8 (August onwards) → season = that year. If month < 8 (before August) → season = year - 1. So a match in May 2018 belongs to the 2017 season (2017/18).

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Season'] = df['Date'].apply(lambda x: x.year if x.month >= 8 else x.year - 1)

print('Date range:', df['Date'].min(), 'to', df['Date'].max())
print()
print('Seasons covered:', sorted(df['Season'].unique()))
print('Total seasons:', df['Season'].nunique())

df = df.drop(columns=['Date'])
print()
print('Shape after dropping Date:', df.shape)

### Output
```
Date range: 2000-08-19 to 2018-05-13

Seasons covered: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
                  2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
Total seasons: 18

Shape after dropping Date: (6840, 35)
```

### Observations
- Dataset covers **18 complete Premier League seasons**: 2000/01 through 2017/18
- 18 seasons × 380 matches per season = 6,840 rows — this confirms the dataset is complete with no missing seasons
- The last match is 13th May 2018 (end of the 2017/18 season)
- Season is now a numeric feature ranging from 2000 to 2017

### Note
- The `Season` feature captures era effects — football in 2000 was played differently to football in 2017 (different rules, tactics, fitness levels)
- This may or may not be useful for the model — feature importance analysis in the modelling notebook will tell us

---
## Step 8 — Convert Matchweek to Integer

**What it does:** Converts the `MW` column from `float64` to `int64`.

**Why:** `MW` represents matchweek (1–38) and should be a whole number. It was stored as float64 likely due to how the original data was merged. Converting it is cleaner and more accurate.

In [ ]:
print('MW dtype before:', df['MW'].dtype)
df['MW'] = df['MW'].astype(int)
print('MW dtype after:', df['MW'].dtype)
print('MW range:', df['MW'].min(), 'to', df['MW'].max())

### Output
```
MW dtype before: float64
MW dtype after: int64
MW range: 1 to 38
```

### Observations
- Matchweek correctly ranges from 1 to 38 (a full Premier League season)
- No data loss from the conversion — all values were already whole numbers stored as floats

---
## Step 9 — Encode the Target Variable

**What it does:** Converts the `FTR` target column from string labels (H/D/A) to numeric values.

**Why:** ML models require numeric targets for classification. The encoding used is:
- `H` = 2 (Home Win)
- `D` = 1 (Draw)
- `A` = 0 (Away Win)

**Why this order?** It creates a natural ordering from worst to best outcome from the home team's perspective (0=loss, 1=draw, 2=win). This is consistent with the encoding that will be used for `skysports_match_stats.csv`.

In [ ]:
target_map = {'H': 2, 'D': 1, 'A': 0}
df['FTR'] = df['FTR'].map(target_map)

print('FTR after encoding:')
print(df['FTR'].value_counts().sort_index())
print()
print('Meaning: 2=Home Win, 1=Draw, 0=Away Win')

### Output
```
FTR after encoding:
0    1913
1    1751
2    3176

Meaning: 2=Home Win, 1=Draw, 0=Away Win
```

### Observations
- All 3 classes present with correct counts
- Class distribution: Home Win 46.4%, Away Win 28.0%, Draw 25.6%
- This imbalance (Home Win is almost twice as common as Draw or Away Win) will need to be considered during modelling

### Notes for Modelling
- A dummy classifier that always predicts Home Win (2) would achieve ~46.4% accuracy — this is our baseline to beat
- Use `class_weight='balanced'` in Logistic Regression and Random Forest to compensate for imbalance
- XGBoost handles this through its internal loss function

---
## Step 10 — Final Check

**What it does:** Verifies the final state of the dataset — shape, column names, data types, missing values, and a preview of the data.

**Why:** Before saving, always confirm everything looks correct. A final check catches any mistakes made during preprocessing.

In [ ]:
print('Final shape:', df.shape)
print()
print('Final columns:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:', df.isnull().sum().sum())
print()
df.head()

### Output
```
Final shape: (6840, 35)

Final columns: ['HomeTeam', 'AwayTeam', 'FTR', 'HTGS', 'ATGS', 'HTGC', 'ATGC',
  'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5',
  'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5',
  'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5',
  'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'Season']

Missing values: 0
```

### Observations
- **6,840 rows preserved** — no data was lost during preprocessing
- **35 columns**: 1 target (`FTR`) + 34 features
- **0 missing values** — clean dataset
- All columns are now numeric — no string values remain
- The dataset is ready for feature engineering or direct model training

### Feature Summary
| Feature Group | Columns | Count |
|---|---|---|
| Team identity | `HomeTeam`, `AwayTeam` | 2 |
| Season context | `Season`, `MW` | 2 |
| Season totals | `HTGS`, `ATGS`, `HTGC`, `ATGC` | 4 |
| Points per game | `HTP`, `ATP` | 2 |
| Last 5 results | `HM1–HM5`, `AM1–AM5` | 10 |
| Form points | `HTFormPts`, `ATFormPts` | 2 |
| Streaks | `HTWinStreak3/5`, `HTLossStreak3/5`, `ATWinStreak3/5`, `ATLossStreak3/5` | 8 |
| Goal difference | `HTGD`, `ATGD` | 2 |
| Comparison | `DiffPts`, `DiffFormPts` | 2 |
| **Total features** | | **34** |

---
## Step 11 — Save the Processed Dataset

**What it does:** Saves the cleaned, preprocessed dataset to the `data/processed/` folder.

**Why:** Separating raw and processed data is good practice. The processed file is what will be loaded in the modelling notebook — we never modify the raw file.

In [ ]:
df.to_csv('../data/processed/premier_league_matches_processed.csv', index=False)
print('Saved to data/processed/premier_league_matches_processed.csv')
print('Shape:', df.shape)

### Output
```
Saved to data/processed/premier_league_matches_processed.csv
Shape: (6840, 35)
```

### Observations
- File saved successfully
- `index=False` prevents pandas from writing an unwanted row index column (avoiding the same `Unnamed: 0` issue we had in the raw file)

---
## Summary of All Preprocessing Steps

| Step | Action | Before | After |
|---|---|---|---|
| 1 | Load data | — | (6840, 40) |
| 2 | Drop useless columns | (6840, 40) | (6840, 37) |
| 3 | Reconstruct FTR (H/D/A) | Binary H/NH | 3-class H/D/A |
| 4 | Drop FTHG/FTAG (leakage) | (6840, 37) | (6840, 35) |
| 5 | Encode form columns | W/D/L/M strings | 3/1/0/0 integers |
| 6 | Encode team names | String names | Integer labels (0–43) |
| 7 | Extract Season, drop Date | Date string | Season integer |
| 8 | Convert MW to int | float64 | int64 |
| 9 | Encode target FTR | H/D/A strings | 2/1/0 integers |
| 10 | Final check | — | (6840, 35), 0 missing |
| 11 | Save to processed/ | — | `premier_league_matches_processed.csv` |

---
## Next Steps
1. Create `cycle1_preprocessing_skysports_match_stats.ipynb` — preprocess the second dataset
2. After both are processed, create `cycle1_modelling.ipynb` — train and evaluate models
3. Compare results across both datasets to decide which to use or whether to combine them